In [1]:
import black
import jupyter_black

jupyter_black.load()

In [2]:
import numpy as np
import openTSNE as TSNE
import matplotlib.pyplot as plt
from dataclasses import dataclass
import importlib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import pandas as pd
import sklearn.datasets
from openTSNE.nearest_neighbors import NNDescent

from sklearn.neighbors import NearestNeighbors

In [3]:
import utils.Utils as Utils
from utils.Utils import GoodnessOfFit

importlib.reload(Utils)

<module 'utils.Utils' from '/Users/a1/Documents/t-SNE-project/utils/Utils.py'>

In [4]:
# set random seed
np.random.seed(42)

#### Set degree of freedeom range (Alphas) in use

In [5]:
ALPHAS = [0.3, 0.6, 0.8, 1, 2, 4, 8, 16, 32, 64, 100]
print(ALPHAS)

[0.3, 0.6, 0.8, 1, 2, 4, 8, 16, 32, 64, 100]


## Swiss-roll dataset

## N-samples = 500

In [6]:
n_samples = 500
sr_points, sr_color = Utils.generate_swiss_roll(
    n_samples=n_samples, noise=0.0, plot="plotly", save_fig=False
)

### Highlight overlapping dots

In [7]:
importlib.reload(Utils)

# Find indices of points that are overlapping in the tsne-space of the swiss roll
overlapping_points = np.where(
    (-5 < sr_points[:, 2]) & (sr_points[:, 2] < 5) & (sr_points[:, 0] > 10)
)

# Get coordinates of overlapping points in the swiss roll
sr_overlapping_points = sr_points[overlapping_points]

fig = Utils.plot_swiss_roll_plotly(sr_points, sr_color, n_samples=n_samples)

# add selected overlapping points overlay
fig.add_trace(
    go.Scatter3d(
        x=sr_overlapping_points[:, 0],
        y=sr_overlapping_points[:, 1],
        z=sr_overlapping_points[:, 2],
        mode="markers",
        marker=dict(
            size=4,
            color="rgba(0,0,0,0)",  # Transparent fill
            line=dict(color="white", width=3),  # Hollow diamond
            symbol="diamond",
            opacity=1,
        ),
        name="Selected overlapping dots",
    )
)
fig.update_layout(scene=dict(aspectmode="cube"), template="plotly_dark")

In [8]:
# compute tsne embedding for the swiss roll with alpha=100
importlib.reload(Utils)
tsne = Utils.compute_tsne_embedding(
    raw_data=sr_points, alphas=[ALPHAS[-1]], dataset_name="swiss_roll_500"
)

Computing the shared affinities...
Computing the PCA initialization...
Computing t-SNE embedding for alpha=100...


In [9]:
fig = Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=tsne,
    labels=sr_color,
    alphas=[ALPHAS[-1]],
    labeled=False,
    width=600,
    height=600,
)

AttributeError: module 'utils.Utils' has no attribute 'plot_TSNE_plotly'

### Locate pre-computed nearest neighbors using NNDescent

In [ ]:
knn_index = NNDescent(data=sr_points, metric="euclidean", k=10)
indices_nndescent, distances_nndescent = knn_index.build()

In [ ]:
selected_point_3D = sr_points[overlapping_points][5]
neigbors_of_5_3D = sr_points[indices_nndescent[overlapping_points][5]]

fig0 = Utils.plot_swiss_roll_plotly(sr_points, sr_color, n_samples=n_samples)
fig0.add_trace(
    go.Scatter3d(
        x=[selected_point_3D[0]],
        y=[selected_point_3D[1]],
        z=[selected_point_3D[2]],
        mode="markers",
        marker=dict(
            size=5,
            color="red",
            symbol="diamond",
            opacity=0.75,
        ),
        name="Selected dot",
    )
)

for i, neighbor in enumerate(neigbors_of_5_3D):
    line_start = (selected_point_3D[0], selected_point_3D[1], selected_point_3D[2])
    line_end = (neighbor[0], neighbor[1], neighbor[2])
    fig0.add_trace(
        go.Scatter3d(
            x=[line_start[0], line_end[0]],
            y=[line_start[1], line_end[1]],
            z=[line_start[2], line_end[2]],
            mode="lines",
            line=dict(color="white", width=3),
            showlegend=(i == 0),  # Show legend only once
            name="Connections to nearest neighbors",
        )
    )
    fig0.add_trace(
        go.Scatter3d(
            x=[neighbor[0]],
            y=[neighbor[1]],
            z=[neighbor[2]],
            mode="markers",
            marker=dict(
                size=5,
                color="rgba(0,0,0,0)",  # Transparent fill
                line=dict(color="white", width=4),  # Hollow diamond
                symbol="diamond",
                opacity=1,
            ),
            name="Nearest neighbors",
            showlegend=(i == 0),  # Show legend only once
        )
    )

fig0.show()

### Locate 10 nearest neighbors in TSNE-space, using the kNN

In [ ]:
overlapping_points_tsne = tsne[0][overlapping_points]

In [ ]:
knn = NearestNeighbors(n_neighbors=10)
knn.fit(tsne[0])
distances, indices = knn.kneighbors(overlapping_points_tsne)
kneighbors_of_5 = tsne[0][indices[5]]

In [ ]:
importlib.reload(Utils)
# select a specific point and its neighbors
selected_point_3D = sr_points[overlapping_points][5]
neighbors_3D = sr_points[indices[5]]
selected_point = (overlapping_points_tsne[5][0], overlapping_points_tsne[5][1])
neighbors = [(x, y) for x, y in kneighbors_of_5]

### PLOT TSNE ###
fig = Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=tsne,
    alphas=[ALPHAS[-1]],
    labeled=False,
    labels=sr_color,
    width=1200,
    height=800,
    show=False,
    grid=(1, 1),
    title="t-SNE embedding of the Swiss Roll dataset. α = 100. N = 500. kNN overlay for a selected point.",
)

# Extract coordinates of the selected point and neighbors


# Plot nearest neighbors as hollow diamonds
fig.add_trace(
    go.Scatter(
        x=[n[0] for n in neighbors],
        y=[n[1] for n in neighbors],
        mode="markers",
        marker=dict(
            size=5,
            color="rgba(0,0,0,0)",  # Transparent fill
            line=dict(color="white", width=2),  # Hollow diamond
            symbol="diamond",
            opacity=0.75,
        ),
        name="10 nearest neighbors",
    ),
    row=1,
    col=1,
)

# Add lines connecting the selected point to its neighbors
for i, neighbor in enumerate(neighbors):
    fig.add_trace(
        go.Scatter(
            x=[selected_point[0], neighbor[0]],
            y=[selected_point[1], neighbor[1]],
            mode="lines",
            line=dict(color="white", width=2),
            showlegend=(i == 0),  # Show legend only once
            name="Connections to nearest neighbors",
        ),
        row=1,
        col=1,
    )

# Highlight the selected dot in red
fig.add_trace(
    go.Scatter(
        x=[selected_point[0]],
        y=[selected_point[1]],
        mode="markers",
        marker=dict(size=6, color="red", symbol="diamond", opacity=0.75),
        name="Selected dot",
    ),
    row=1,
    col=1,
)


# Display the figure
fig.show()


### PLOT SWISS ROLL ###


fig0 = Utils.plot_swiss_roll_plotly(
    sr_points,
    sr_color,
    n_samples=n_samples,
    title="Swiss Roll dataset. N = 500. kNN overlay from t-SNE space for a selected point.",
)
fig0.add_trace(
    go.Scatter3d(
        x=[selected_point_3D[0]],
        y=[selected_point_3D[1]],
        z=[selected_point_3D[2]],
        mode="markers",
        marker=dict(size=6, color="red", symbol="diamond", opacity=0.75),
        name="Selected point",
    )
)

for i, neighbor in enumerate(neighbors_3D):
    line_start = (selected_point_3D[0], selected_point_3D[1], selected_point_3D[2])
    line_end = (neighbor[0], neighbor[1], neighbor[2])
    fig0.add_trace(
        go.Scatter3d(
            x=[line_start[0], line_end[0]],
            y=[line_start[1], line_end[1]],
            z=[line_start[2], line_end[2]],
            mode="lines",
            line=dict(color="white", width=3),
            showlegend=(i == 0),  # Show legend only once
            name="Connections to nearest neighbors",
        )
    )

    fig0.add_trace(
        go.Scatter3d(
            x=[neighbor[0]],
            y=[neighbor[1]],
            z=[neighbor[2]],
            mode="markers",
            marker=dict(
                size=5,
                color="rgba(0,0,0,0)",  # Transparent fill
                line=dict(color="white", width=4),  # Hollow diamond
                symbol="diamond",
                opacity=1,
            ),
            name="Nearest neighbors",
            showlegend=(i == 0),  # Show legend only once
        )
    )


fig0.show()

In [ ]:
importlib.reload(Utils)
tsne_results_sr500 = Utils.compute_tsne_embedding(raw_data=sr_points, alphas=ALPHAS)

In [ ]:
importlib.reload(Utils)

In [ ]:
# fig, ax = plt.subplots(2, 5, figsize=(20, 8))
Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=tsne_results_sr500,
    labels=sr_color,
    alphas=ALPHAS,
    labeled=False,
    show=False,
    display_metrics=True,
)

In [ ]:
importlib.reload(Utils)
results_sr500 = Utils.compute_goodness_of_fit(
    raw_data=sr_points, tsne_data_list=tsne_results_sr500, dofs=ALPHAS, plot=True
)

## N-samples = 1000

In [ ]:
n_samples = 1000
sr_points, sr_color = Utils.generate_swiss_roll(
    n_samples=n_samples, noise=0.0, plot="plotly", save_fig=False
)

In [ ]:
tsne_results_sr1000 = []
for α in ALPHAS:
    tsne = TSNE.TSNE(
        perplexity=50,
        n_jobs=-1,
        random_state=42,
        dof=α,
        verbose=True,
    )
    tsne_result = tsne.fit(sr_points)
    tsne_results_sr1000.append(tsne_result)

In [ ]:
importlib.reload(Utils)
Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=tsne_results_sr1000,
    labels=sr_color,
    alphas=ALPHAS,
    display_metrics=True,
    labeled=False,
)

In [ ]:
results_sr1000 = Utils.compute_goodness_of_fit(
    raw_data=sr_points, tsne_data_list=tsne_results_sr1000, dofs=ALPHAS, plot=True
)

### N-samples = 5000

In [ ]:
importlib.reload(Utils)

In [ ]:
n_samples = 5000
sr_points, sr_color = Utils.generate_swiss_roll(
    n_samples=5000, noise=0.0, plot="plotly"
)

In [ ]:
tsne_results_sr5000 = Utils.compute_tsne_embedding(raw_data=sr_points, alphas=ALPHAS)

In [ ]:
Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=tsne_results_sr5000,
    labels=sr_color,
    alphas=ALPHAS,
    display_metrics=True,
)

In [ ]:
results_sr5000 = Utils.compute_goodness_of_fit(
    raw_data=sr_points, tsne_data_list=tsne_results_sr5000, dofs=ALPHAS, plot=True
)

In [ ]:
fig = make_subplots(1, 2, subplot_titles=("KL Divergence", "kNN Recall"))

fig.add_trace(
    go.Scatter(
        x=[500, 1000, 5000],
        y=[
            results_sr500.optimal_alpha_KL,
            results_sr1000.optimal_alpha_KL,
            results_sr5000.optimal_alpha_KL,
        ],
        mode="lines+markers",
        name="KL Divergence",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=[500, 1000, 5000],
        y=[
            results_sr500.optimal_alpha_kNN_recall,
            results_sr1000.optimal_alpha_kNN_recall,
            results_sr5000.optimal_alpha_kNN_recall,
        ],
        mode="lines+markers",
        name="kNN Recall",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Number of Samples", tickvals=[500, 1000, 5000])
fig.update_yaxes(title_text="optimal α value", tickvals=ALPHAS.round(2))

fig.update_layout(
    width=1200,
    height=400,
    title_text="Optimal α value for t-SNE embeddings of the Swiss Roll dataset with different number of samples",
    template="plotly_dark",
)

In [ ]:
def get_tsne_results_for_optimal_alpha(tsne_results, results, method):

    if method == "KL":
        optimal_alpha_idx = np.where(results.alphas == results.optimal_alpha_KL)[0][0]
    elif method == "kNN":
        optimal_alpha_idx = np.where(
            results.alphas == results.optimal_alpha_kNN_recall
        )[0][0]
    return tsne_results[optimal_alpha_idx]


chosen_tsnes = [
    tsne_results_sr500[2],
    tsne_results_sr1000[2],
    tsne_results_sr5000[2],
    get_tsne_results_for_optimal_alpha(tsne_results_sr500, results_sr500, "KL"),
    get_tsne_results_for_optimal_alpha(tsne_results_sr1000, results_sr1000, "KL"),
    get_tsne_results_for_optimal_alpha(tsne_results_sr5000, results_sr5000, "KL"),
    tsne_results_sr500[-1],
    tsne_results_sr1000[-1],
    tsne_results_sr5000[-1],
]
chosen_alphas = [
    ALPHAS[2],
    ALPHAS[2],
    ALPHAS[2],
    results_sr500.optimal_alpha_KL,
    results_sr1000.optimal_alpha_KL,
    results_sr5000.optimal_alpha_KL,
    ALPHAS[-1],
    ALPHAS[-1],
    ALPHAS[-1],
]

importlib.reload(Utils)
fig = Utils.plot_TSNE_plotly(
    raw_data=sr_points,
    tsne_results=chosen_tsnes,
    labels=sr_color,
    alphas=chosen_alphas,
    display_metrics=False,
    grid=(3, 3),
    title="Embeddings of the Swiss Roll dataset with different N and α values",
    width=1200,
    height=1200,
    show=False,
)

fig.add_annotation(
    x=0.5,
    y=0.72,
    xref="paper",
    yref="paper",
    text="t-SNE",
    showarrow=False,
    font=dict(size=20),
)

fig.add_annotation(
    x=0.5,
    y=0.35,
    xref="paper",
    yref="paper",
    text="optimal α",
    showarrow=False,
    font=dict(size=20),
)
fig.add_annotation(
    x=0.5,
    y=0,
    xref="paper",
    yref="paper",
    text="SNE",
    showarrow=False,
    font=dict(size=20),
)

## Digits dataset

In [ ]:
digits = sklearn.datasets.load_digits(n_class=10)
digit_colors = sns.color_palette("Spectral", as_cmap=True)
X, y = digits.data, digits.target
print(y)
n_samples, n_features = X.shape
n_neighbors = 30

In [ ]:
fig, axs = plt.subplots(nrows=10, ncols=10, figsize=(6, 6))
for idx, ax in enumerate(axs.ravel()):
    ax.imshow(X[idx].reshape((8, 8)), cmap="grey")
    ax.axis("off")
_ = fig.suptitle("A selection from the 64-dimensional digits dataset", fontsize=16)

In [ ]:
# X = MinMaxScaler().fit_transform(X)
digits_tsne = TSNE.TSNE(perplexity=30).fit(X)

In [ ]:
importlib.reload(Utils)
Utils.plot_TSNE_plotly(
    raw_data=X,
    tsne_results=[digits_tsne],
    labels=y,
    alphas=[1],
    display_metrics=False,
    title="t-SNE embedding of the digits dataset",
    show=False,
    width=1000,
    height=800,
    grid=(1, 1),
    colorscale="Rainbow",
)

In [ ]:
# add labels to the embedding
digits_tsne = np.hstack([digits_tsne, y[:, np.newaxis]])
print(digits_tsne)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
for n in range(int(digits_tsne[:, 2].max()) + 1):
    ax.scatter(
        digits_tsne[digits_tsne[:, 2] == n, 0],
        digits_tsne[digits_tsne[:, 2] == n, 1],
        s=50,
        alpha=0.8,
        cmap=digit_colors,
        label=f"digit {n}",
    )
sns.despine(left=True, bottom=True)
ax.set_title("Digits dataset in TSNE space")
ax.set_xticks([])
ax.set_yticks([])
ax.legend()

In [ ]:
digit_tsne_results = Utils.compute_tsne_embedding(raw_data=X, alphas=ALPHAS)

In [ ]:
print(y)

In [ ]:
importlib.reload(Utils)
fig = Utils.plot_TSNE_plotly(
    raw_data=X,
    tsne_results=digit_tsne_results,
    labels=y,
    alphas=ALPHAS,
    display_metrics=True,
    width=1800,
    height=800,
    custom_palette=False,
    colorscale="Jet",
)

In [ ]:
results_digits = Utils.compute_goodness_of_fit(
    raw_data=X, tsne_data_list=digit_tsne_results, dofs=ALPHAS, plot=True
)

In [ ]:
from keras.datasets import mnist

In [ ]:
(train_X, train_y), (test_X, test_y) = mnist.load_data()

In [ ]:
current_digit = 0
for i in range(9):
    y_index = np.where(train_y == current_digit)[0][i]
    plt.subplot(330 + 1 + i)
    plt.imshow(train_X[y_index], cmap=plt.get_cmap("gray"))
    current_digit += 1
    sns.despine(left=True, bottom=True)

In [ ]:
def plot_digits_plotly(data, labels, title):

    fig = make_subplots(2, 5, subplot_titles=[f"Digit {i}" for i in range(10)])
    current_digit = 0

    for i in range(10):
        # Get the first image of the current digit
        image_index = np.where(labels == current_digit)[0][0]
        image = data[image_index]
        image = np.flipud(image)

        # Add image to the subplot using Heatmap for grayscale images
        fig.add_trace(
            go.Heatmap(
                z=image,
                colorscale="Greys",
                showscale=False,
            ),
            row=(i // 5) + 1,
            col=(i % 5) + 1,
        )
        current_digit += 1

    # Update layout of the figure
    fig.update_layout(
        height=600, width=1000, title_text="Digits Dataset", template="plotly_dark"
    )

    # Show the figure
    return fig

In [ ]:
plot_digits_plotly(train_X, train_y, "Digits Dataset")

In [ ]:
n_samples = 1000
train_X_reduced = train_X[:n_samples]
train_y_reduced = train_y[:n_samples]
for i in range(10):
    assert i in train_y_reduced

In [ ]:
plot_digits_plotly(train_X_reduced, train_y_reduced, "Digits Dataset")

In [ ]:
# reshape for tsne
train_X_reduced = train_X_reduced.reshape(n_samples, -1)
train_X_reduced.shape

tsne_mnist_reduced = Utils.compute_tsne_embedding(
    raw_data=train_X_reduced, alphas=ALPHAS
)

In [ ]:
Utils.plot_TSNE_plotly(
    raw_data=train_X_reduced,
    tsne_results=tsne_mnist_reduced,
    labels=train_y_reduced,
    alphas=ALPHAS,
    display_metrics=True,
    width=1800,
    height=800,
    colorscale="Jet",
    labeled=True,
    show=False,
)

In [ ]:
x = train_X.reshape(train_X.shape[0], -1)
print(x.shape)
tsne = TSNE.TSNE(perplexity=30, n_jobs=-1, random_state=42, verbose=True).fit(x)

In [ ]:
plt.figure(figsize=(10, 10))
Utils.plot_TSNE_plotly(
    raw_data=x,
    tsne_results=[tsne],
    labels=train_y,
    alphas=[1],
    display_metrics=False,
    title="t-SNE embedding of the MNIST dataset",
    show=False,
    width=1000,
    height=800,
    grid=(1, 1),
    colorscale="Jet",
)

In [ ]:
tsne_results_mnist = Utils.compute_tsne_embedding(raw_data=x, alphas=[1])

In [ ]:
Utils.plot_TSNE_plotly(
    raw_data=x,
    tsne_results=tsne_results_mnist,
    labels=train_y,
    alphas=[1],
    display_metrics=False,
    title="t-SNE embedding of the MNIST dataset",
    show=False,
    width=1000,
    height=800,
    grid=(1, 1),
    colorscale="Jet",
)

In [ ]:
importlib.reload(Utils)
x = train_X.reshape(train_X.shape[0], -1)
tsne_mnist_all_alphas = Utils.compute_tsne_embedding(
    raw_data=x, alphas=ALPHAS, dataset_name="mnist_all_datapoints_all_alphas"
)

In [ ]:
Utils.plot_TSNE_plotly(
    raw_data=x,
    tsne_results=tsne_mnist_all_alphas,
    labels=train_y,
    alphas=ALPHAS,
    display_metrics=False,
    width=1800,
    height=800,
    colorscale="Jet",
    labeled=False,
    show=True,
)

In [ ]:
mnist_per_digit = []
for i in range(10):

    label_i = np.where(train_y == i)[0][0]
    mnist_per_digit.append(train_X[label_i].reshape(train_X.shape[0], -1))

In [ ]:
tsnes_mnist_per_digit_per_alpha = []
for i in range(len(mnist_per_digit)):
    tsnes = Utils.compute_tsne_embedding(raw_data=mnist_per_digit[i], alphas=ALPHAS)
    tsnes_mnist_per_digit_per_alpha.append(tsnes)
